In [ ]:
# =========================
# nb_ingest_wh_ireland_v2
# Stage 1: Faithful Source Ingestion
# =========================
"""
Purpose: Provide a faithful representation of WH Ireland source files
         in Fabric for Stage 2 processing.

File Format: Excel (.xlsx) files
Strategy: Read all sheets, attempt to detect headers, preserve all data
"""

# ---------- Establish Workspace Context ----------
# CRITICAL: Required when notebook is in a subfolder
# This ensures Fabric's lakehouse context is properly initialized
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")
    print("[ACTION] Ensure lakehouse is attached in Fabric notebook UI")

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

dfm_id = "wh_ireland"
dfm_name = "WH Ireland"

print(f"[INFO] Starting WH Ireland ingestion for period={period}, run_id={run_id}")

In [ ]:
# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
from datetime import datetime
import re

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
print(f"[INFO] Landing path: {landing_path}")

In [ ]:
# ---------- Helper Functions ----------

def normalize_column_name(col_name):
    """Normalize column names: remove special chars, lowercase, replace spaces with underscores."""
    normalized = str(col_name).strip().lower()
    normalized = re.sub(r'[^\w\s]', '', normalized)
    normalized = re.sub(r'\s+', '_', normalized)
    normalized = normalized.strip('_')
    # Handle unnamed columns
    if normalized.startswith('unnamed'):
        return normalized
    return normalized if normalized else 'unnamed_col'

def read_excel_sheet(file_path, sheet_name, source_file_name):
    """
    Read Excel sheet with automatic header detection.
    Tries to find the header row by looking for rows with non-empty, meaningful column names.
    
    Returns: Spark DataFrame with all columns as strings, or None if failed
    """
    try:
        # Try reading with different header row positions
        for header_row in range(0, 5):  # Try first 5 rows as potential header
            try:
                # Read the sheet
                pdf = pd.read_excel(
                    file_path,
                    sheet_name=sheet_name,
                    header=header_row,
                    dtype=str,
                    engine='openpyxl'
                )
                
                if pdf is None or len(pdf) == 0:
                    continue
                
                # Check if we have a reasonable header
                # Good headers should have mostly non-empty, unique values
                cols_str = [str(c).strip() for c in pdf.columns]
                non_empty = [c for c in cols_str if c and not c.startswith('Unnamed:')]
                
                if len(non_empty) >= len(cols_str) * 0.5:  # At least 50% named columns
                    # Normalize column names
                    pdf.columns = [normalize_column_name(c) for c in pdf.columns]
                    
                    # Remove completely empty rows
                    pdf = pdf.dropna(how='all')
                    
                    if len(pdf) == 0:
                        continue
                    
                    # Replace NaN with empty string
                    pdf = pdf.fillna('')
                    
                    # Convert to Spark DataFrame
                    sdf = spark.createDataFrame(pdf)
                    
                    # Add source tracking
                    sdf = (
                        sdf.withColumn("source_file", F.lit(source_file_name))
                           .withColumn("source_sheet", F.lit(str(sheet_name)))
                    )
                    
                    print(f"[INFO] Successfully read sheet '{sheet_name}' (header at row {header_row}): {sdf.count()} rows, {len(sdf.columns)-2} columns")
                    return sdf
                    
            except Exception as e:
                continue
        
        # If automatic detection failed, read without header and preserve everything
        print(f"[WARNING] Could not auto-detect header for sheet '{sheet_name}', reading raw")
        pdf = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            header=None,
            dtype=str,
            engine='openpyxl'
        )
        
        if pdf is None or len(pdf) == 0:
            return None
        
        # Create generic column names
        pdf.columns = [f'col_{i}' for i in range(len(pdf.columns))]
        pdf = pdf.fillna('')
        
        # Remove completely empty rows
        pdf = pdf[pdf.apply(lambda x: x.str.strip().str.len().sum(), axis=1) > 0]
        
        if len(pdf) == 0:
            return None
        
        sdf = spark.createDataFrame(pdf)
        sdf = (
            sdf.withColumn("source_file", F.lit(source_file_name))
               .withColumn("source_sheet", F.lit(str(sheet_name)))
               .withColumn("header_detected", F.lit(False))
        )
        
        print(f"[INFO] Read sheet '{sheet_name}' as raw data: {sdf.count()} rows")
        return sdf
        
    except Exception as e:
        print(f"[ERROR] Failed to read sheet '{sheet_name}': {str(e)}")
        return None

def add_row_identifiers(df, sheet_prefix):
    """Add unique row identifiers to each record."""
    w = Window.partitionBy("source_sheet").orderBy(F.monotonically_increasing_id())
    return (
        df.withColumn("_row_num", F.row_number().over(w))
          .withColumn(
              "source_row_id",
              F.concat_ws(":", 
                  F.col("source_file"),
                  F.col("source_sheet"),
                  F.lit(sheet_prefix),
                  F.col("_row_num").cast("string")
              )
          )
          .drop("_row_num")
    )

def add_metadata(df):
    """Add standard metadata columns to the dataframe."""
    return (
        df.withColumn("period", F.lit(period))
          .withColumn("run_id", F.lit(run_id))
          .withColumn("dfm_id", F.lit(dfm_id))
          .withColumn("dfm_name", F.lit(dfm_name))
          .withColumn("ingested_at", F.current_timestamp())
    )

print("[INFO] Helper functions loaded")

In [ ]:
# ---------- Discover Files ----------
print("[STEP 1] Discovering source files...")

try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
    excel_files = [f for f in entries if f.name.lower().endswith(('.xlsx', '.xls'))]
    print(f"[INFO] Found {len(excel_files)} Excel file(s)")
    for f in excel_files:
        print(f"  - {f.name}")
except Exception as e:
    print(f"[ERROR] Could not access landing path: {str(e)}")
    mssparkutils.notebook.exit("NO_FILES")

if len(excel_files) == 0:
    print("[ERROR] No Excel files found")
    mssparkutils.notebook.exit("NO_FILES")

In [ ]:
# ---------- Read Excel Files and Sheets ----------
print("[STEP 2] Reading Excel files and sheets...")

all_sheets_data = []
file_summary = []

for excel_file in excel_files:
    print(f"\n[INFO] Processing: {excel_file.name}")
    
    try:
        # Get all sheet names
        xl_file = pd.ExcelFile(excel_file.path, engine='openpyxl')
        sheet_names = xl_file.sheet_names
        print(f"[INFO] Found {len(sheet_names)} sheet(s): {', '.join(sheet_names)}")
        
        for sheet_name in sheet_names:
            print(f"[INFO] Reading sheet: {sheet_name}")
            df_sheet = read_excel_sheet(excel_file.path, sheet_name, excel_file.name)
            
            if df_sheet:
                df_sheet = add_row_identifiers(df_sheet, "ROW")
                all_sheets_data.append(df_sheet)
                
                file_summary.append({
                    'file': excel_file.name,
                    'sheet': sheet_name,
                    'rows': df_sheet.count(),
                    'columns': len([c for c in df_sheet.columns if c not in ['source_file', 'source_sheet', 'source_row_id']])
                })
            else:
                print(f"[WARNING] Could not read sheet: {sheet_name}")
        
    except Exception as e:
        print(f"[ERROR] Failed to process {excel_file.name}: {str(e)}")

if len(all_sheets_data) == 0:
    print("[ERROR] No data could be read from any Excel files")
    mssparkutils.notebook.exit("NO_DATA")

print(f"\n[INFO] Successfully read {len(all_sheets_data)} sheet(s) across {len(excel_files)} file(s)")

In [ ]:
# ---------- Combine All Sheets ----------
print("[STEP 3] Combining data from all sheets...")

# Combine all sheets - use unionByName to handle different schemas
df_combined = all_sheets_data[0]
for df in all_sheets_data[1:]:
    df_combined = df_combined.unionByName(df, allowMissingColumns=True)

total_rows = df_combined.count()
print(f"[INFO] Combined data: {total_rows} total rows")

# Show summary
print("\n[INFO] File and sheet summary:")
for summary in file_summary:
    print(f"  {summary['file']} / {summary['sheet']}: {summary['rows']} rows, {summary['columns']} columns")

In [ ]:
# ---------- Profile the Data ----------
print("\n" + "=" * 80)
print("[DATA PROFILE] Source File Statistics")
print("=" * 80)

print(f"\nTotal Excel files: {len(excel_files)}")
print(f"Total sheets processed: {len(file_summary)}")
print(f"Total rows: {total_rows}")

# Get distinct sheets
distinct_sheets = df_combined.select("source_file", "source_sheet").distinct()
print(f"\nSheets by file:")
distinct_sheets.orderBy("source_file", "source_sheet").show(truncate=False)

# Check for common columns that might indicate data structure
all_columns = [c for c in df_combined.columns if c not in ['source_file', 'source_sheet', 'source_row_id', 'period', 'run_id', 'dfm_id', 'dfm_name', 'ingested_at']]
print(f"\nColumn count: {len(all_columns)}")

# Show sample columns (first 10)
if len(all_columns) > 0:
    print(f"Sample columns: {', '.join(all_columns[:10])}")
    if len(all_columns) > 10:
        print(f"... and {len(all_columns) - 10} more")

print("=" * 80 + "\n")

In [ ]:
# ---------- Add Metadata Columns ----------
print("[STEP 4] Adding metadata columns...")

df_final = add_metadata(df_combined)
rows_ready = df_final.count()
print(f"[INFO] Metadata added: {rows_ready} rows ready")

In [ ]:
# ---------- Write to Staging Table ----------
print("[STEP 5] Writing to staging table...")

table_name = "stage1_wh_ireland_raw"
print(f"[INFO] Writing all sheets to {table_name}...")

try:
    df_final.write.mode("append").saveAsTable(table_name)
    rows_written = df_final.count()
    print(f"[SUCCESS] Written {rows_written} rows to {table_name}")
except Exception as e:
    print(f"[ERROR] Failed to write to table: {str(e)}")
    rows_written = 0

In [ ]:
# ---------- Write Audit Log ----------
print("[STEP 6] Writing audit log...")

try:
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, 
         parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{run_id}', '{period}', '{dfm_id}', {len(excel_files)}, {rows_written}, 
         0, 0, 'OK', current_timestamp(), current_timestamp())
    """)
    print("[SUCCESS] Audit log written")
except Exception as e:
    print(f"[WARNING] Could not write audit log: {str(e)}")

In [ ]:
# ---------- Summary ----------
print("\n" + "=" * 80)
print("[INGESTION COMPLETE]")
print("=" * 80)
print(f"Period: {period}")
print(f"Run ID: {run_id}")
print(f"DFM: {dfm_name}")
print(f"Files processed: {len(excel_files)}")
for f in excel_files:
    print(f"  - {f.name}")
print(f"Sheets processed: {len(file_summary)}")
print(f"Total rows ingested: {rows_written}")
print(f"Staging table: {table_name}")
print(f"Status: OK")
print("=" * 80)

mssparkutils.notebook.exit("OK")